In [64]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

In [65]:
columns = [ "age", "workclass", "fnlwgt", "education", "education_num", "marital_status", "occupation", 
           "relationship", "race", "sex", "capital_gain", "capital_loss", "hours_per_week", "native_country", "income" ] 

In [66]:
# Load the Adult dataset from the file "adult.data". 
# header=None → The dataset does not contain column names in the first row.
# so pandas should NOT treat the first row as a header.
# names=columns → Assign the predefined list of column names to the DataFrame. 
# na_values="?" → Convert any "?" in the dataset into proper missing values (NaN). 
# skipinitialspace=True → Remove any extra spaces after commas to clean the data.

df_train = pd.read_csv("adult.data", header=None, names=columns, na_values="?", skipinitialspace=True) 

In [67]:
df_train.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [68]:
#df_test = pd.read_csv("adult_test.csv", header=None, names=columns, na_values="?", skipinitialspace=True)
df_test = pd.read_csv("adult.test", header=None, names=columns, na_values="?", skipinitialspace=True)

In [69]:
df_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,|1x3 Cross validator,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,25,Private,226802.0,11th,7.0,Never-married,Machine-op-inspct,Own-child,Black,Male,0.0,0.0,40.0,United-States,<=50K.
2,38,Private,89814.0,HS-grad,9.0,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K.
3,28,Local-gov,336951.0,Assoc-acdm,12.0,Married-civ-spouse,Protective-serv,Husband,White,Male,0.0,0.0,40.0,United-States,>50K.
4,44,Private,160323.0,Some-college,10.0,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688.0,0.0,40.0,United-States,>50K.


In [70]:
df_train["source"] = "train"
df_test["source"] = "test"


In [71]:
df_train_test = pd.concat([df_train, df_test], ignore_index=True)


In [72]:
df_train_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source
0,39,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K,train
1,50,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K,train
2,38,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K,train
3,53,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K,train
4,28,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K,train


In [73]:
df_train_test.tail()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source
48838,39,Private,215419.0,Bachelors,13.0,Divorced,Prof-specialty,Not-in-family,White,Female,0.0,0.0,36.0,United-States,<=50K.,test
48839,64,NaN,321403.0,HS-grad,9.0,Widowed,NaN,Other-relative,Black,Male,0.0,0.0,40.0,United-States,<=50K.,test
48840,38,Private,374983.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Husband,White,Male,0.0,0.0,50.0,United-States,<=50K.,test
48841,44,Private,83891.0,Bachelors,13.0,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455.0,0.0,40.0,United-States,<=50K.,test
48842,35,Self-emp-inc,182148.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,60.0,United-States,>50K.,test


In [74]:
df_train_test['income'].unique()

array(['<=50K', '>50K', nan, '<=50K.', '>50K.'], dtype=object)

In [75]:
df_train_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48843 entries, 0 to 48842
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             48843 non-null  object 
 1   workclass       46043 non-null  object 
 2   fnlwgt          48842 non-null  float64
 3   education       48842 non-null  object 
 4   education_num   48842 non-null  float64
 5   marital_status  48842 non-null  object 
 6   occupation      46033 non-null  object 
 7   relationship    48842 non-null  object 
 8   race            48842 non-null  object 
 9   sex             48842 non-null  object 
 10  capital_gain    48842 non-null  float64
 11  capital_loss    48842 non-null  float64
 12  hours_per_week  48842 non-null  float64
 13  native_country  47985 non-null  object 
 14  income          48842 non-null  object 
 15  source          48843 non-null  object 
dtypes: float64(5), object(11)
memory usage: 6.0+ MB


In [76]:
df_train_test.describe()

,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,4.884200e+04,48842.000000,48842.000000,48842.000000,48842.000000
mean,1.896641e+05,10.078089,1079.067626,87.502314,40.422382
std,1.056040e+05,2.570973,7452.019058,403.004552,12.391444
min,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,1.175505e+05,9.000000,0.000000,0.000000,40.000000
50%,1.781445e+05,10.000000,0.000000,0.000000,40.000000
75%,2.376420e+05,12.000000,0.000000,0.000000,45.000000
max,1.490400e+06,16.000000,99999.000000,4356.000000,99.000000


In [77]:
df_train_test.isnull().sum()

age                  0
workclass         2800
fnlwgt               1
education            1
education_num        1
marital_status       1
occupation        2810
relationship         1
race                 1
sex                  1
capital_gain         1
capital_loss         1
hours_per_week       1
native_country     858
income               1
source               0
dtype: int64

In [78]:
df_train_test['education'].unique()


array(['Bachelors', 'HS-grad', '11th', 'Masters', '9th', 'Some-college',
       'Assoc-acdm', 'Assoc-voc', '7th-8th', 'Doctorate', 'Prof-school',
       '5th-6th', '10th', '1st-4th', 'Preschool', '12th', nan],
      dtype=object)

In [79]:
df_train_test['income'].unique()


array(['<=50K', '>50K', nan, '<=50K.', '>50K.'], dtype=object)

In [80]:
df_train_test[['education', 'income']].nunique()


education    16
income        4
dtype: int64

In [81]:
df_train_test['income'].value_counts()


income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64

In [82]:
df_train_test['income'] = df_train_test['income'].str.replace('.', '', regex=False).str.strip()


In [83]:
df_train_test['income'].value_counts()


income
<=50K    37155
>50K     11687
Name: count, dtype: int64

<span style="color: blue;"> 
Hypothesis Question 1:
Does education level affect the probability of earning more than 50K?

Variables:
Education → Categorical (16 levels)

Income Class → Binary categorical (>50K, ≤50K)

Hypotheses:
Null Hypothesis (H₀):
Education level does not affect the probability of earning >50K.  
Income distribution is the same across all education categories.

Alternative Hypothesis (H₁):
Higher education levels increase the probability of earning >50K.  
Income distribution differs across education categories. </span>

In [84]:
import scipy.stats as stats


# Create contingency table
table = pd.crosstab(df_train_test['education'], df_train_test['income'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)


Chi-square: 6537.972961360963
Degrees of freedom: 15
p-value: 0.0


<span style="color: blue;">Interpretation: “The chi‑square test produced a p‑value smaller than the minimum representable value in Python (displayed as 0.0), indicating an extremely strong and statistically significant association between education level and income class.” </span>

<span style="color: blue;">Type I Error (False Positive) : reject H₀ even though it is actually true , We conclude that education affects income, even if in reality education has no real effect on income. </span>

<span style="color: blue;">Type II Error (False Negative): fail to reject H₀ even though H₁ is true, We conclude that education does not affect income, even though in reality higher education does increase the probability of earning >50K. </span>

<span style="color: blue;">Education is  associated with income </span>

In [85]:
df_train_test['workclass'].unique()


array(['State-gov', 'Self-emp-not-inc', 'Private', 'Federal-gov',
       'Local-gov', nan, 'Self-emp-inc', 'Without-pay', 'Never-worked'],
      dtype=object)

<span style="color: blue;">Hypothesis Question 2  </span>
<span style="color: blue;">Does Workclass affect the probability of earning more than 50K?  </span>
<span style="color: blue;">Variables: Workclass → Categorical (9 levels, including missing)  </span>

<span style="color: blue;">Income Class → Binary categorical (>50K, ≤50K) </span>

<span style="color: blue;">Hypothesis: </span>
<span style="color: blue;">Null Hypothesis (H₀): </span>
<span style="color: blue;">Workclass does not affect income classification. </span>
<span style="color: blue;">i.e.Income distribution is the same across all workclass categories. </span>

<span style="color: blue;">Alternative Hypothesis (H₁): </span>
<span style="color: blue;">Workclass significantly affects income classification. </span>
<span style="color: blue;">i.e.Income distribution differs across workclass categories. </span>

In [86]:
# Contingency table
table = pd.crosstab(df_train_test['workclass'], df_train_test['income'])

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)


Chi-square: 1238.990039222109
Degrees of freedom: 7
p-value: 2.6121798441462815e-263


<span style="color: blue;"> Interpretation:
p-value  is extremely small , i.e. We reject the null hypothesis (H₀). There is a highly significant association between workclass and income class.
Different workclass categories have different probabilities of earning >50K.  </Span>

<span style="color: blue;">People working in certain workclasses (e.g., Federal-gov, Self-emp-inc) are more likely to earn >50K, while others (e.g., Without-pay, Never-worked) are far less likely. </Span>

<span style="color: blue;">Type I Error (False Positive): Rejecting H₀ when H₀ is actually true, We  conclude that workclass affects income, even though in reality workclass has no effect on income, i.e.  workclass matters for income, but it actually doesn’t. </Span>

<span style="color: blue;">Type II Error (False Negative): Failing to reject H₀ when H₁ is actually true, We conclude that workclass does NOT affect income, even though in reality workclass DOES influence the probability of earning >50K, i.e. we miss a real effect — workclass actually matters, but you fail to detect it. </Span>

<span style="color: blue;"> Hypothesis Question 3 Do high‑education individuals working normal hours have a higher probability of earning >50K than low‑education individuals working long hours? </span>

<span style="color: blue;">Null Hypothesis (H₀):  Low‑education individuals working long hours have the same probability of earning >50K as high‑education individuals working normal hours. </span>

<span style="color: blue;">Alternative Hypothesis (H₁):  High‑education individuals working normal hours have a higher probability of earning >50K than low‑education individuals working long hours. </span>

In [87]:
from statsmodels.stats.proportion import proportions_ztest

# 1. Define education groups
low_edu = [
    'Preschool','1st-4th','5th-6th','7th-8th',
    '9th','10th','11th','12th','HS-grad'
]

high_edu = [
    'Bachelors','Prof-school','Assoc-acdm',
    'Assoc-voc','Masters','Doctorate'
]

# 2. Define hours groups
high_hours = df_train_test['hours_per_week'] > 45          # long hours
normal_hours = df_train_test['hours_per_week'].between(35, 45, inclusive='both')

# 3. Define Group A: Low education + High hours
group_A = df_train_test[
    df_train_test['education'].isin(low_edu) & high_hours
]

# 4. Define Group B: High education + Normal hours
group_B = df_train_test[
    df_train_test['education'].isin(high_edu) & normal_hours
]

# 5. Check sizes
print("Group A size:", len(group_A))
print("Group B size:", len(group_B))


Group A size: 3866
Group B size: 9222


In [88]:
# 1. Count successes (>50K) and totals in each group
group_A_high = np.sum(group_A['income'] == '>50K')
group_A_total = len(group_A)

group_B_high = np.sum(group_B['income'] == '>50K')
group_B_total = len(group_B)

print("Group A: >50K =", group_A_high, "out of", group_A_total)
print("Group B: >50K =", group_B_high, "out of", group_B_total)

# 2. Prepare inputs for two-proportion z-test
counts = np.array([group_A_high, group_B_high])   # successes
nobs   = np.array([group_A_total, group_B_total]) # totals

# 3. Run one-sided test: H1: p_B > p_A  → equivalently p_A < p_B → 'smaller'
stat, p_value = proportions_ztest(counts, nobs, alternative='smaller')

print("Z-statistic:", stat)
print("p-value:", p_value)


Group A: >50K = 944 out of 3866
Group B: >50K = 3596 out of 9222
Z-statistic: -15.98267950707618
p-value: 8.436706557005271e-58


<span style="color: blue;">Interpretaion: The p-value is far more less than 0.05
<span style="color: blue;">This means the difference between the two groups is not due to random chance.
<span style="color: blue;">We reject the null hypothesis (H₀) , High‑education individuals working normal hours have a significantly higher 
probability of earning >50K than low‑education individuals working long hours. </span>

<span style="color: blue;">Type I Error (False Positive)
we conclude Group B > Group A when they are actually equal.  
when in reality both groups have the same probability. 
i.e. we think education + normal hours matters, but it actually doesn’t. </span>

<span style="color: blue;">Type II Error (False Negative)
we conclude that there is no difference between the two groups  
when in reality Group B truly has a higher probability of earning >50K.
i.e. we miss a real effect. </span>

<span style="color: blue;">Hypothesis Question 4:
Does the combination of occupation and hours‑per‑week affect the probability of earning >50K? </span>

<span style="color: blue;">Null Hypothesis (H₀):
Occupation and hours‑per‑week have no effect on income classification.
Income distribution is the same across all occupation × hours groups. </span>

<span style="color: blue;">Alternative Hypothesis (H₁):
Certain occupations combined with longer working hours increase the probability of earning >50K.
Income distribution differs across occupation × hours groups. </span>

In [89]:
sorted(df_train_test['occupation'].dropna().unique())


['Adm-clerical',
 'Armed-Forces',
 'Craft-repair',
 'Exec-managerial',
 'Farming-fishing',
 'Handlers-cleaners',
 'Machine-op-inspct',
 'Other-service',
 'Priv-house-serv',
 'Prof-specialty',
 'Protective-serv',
 'Sales',
 'Tech-support',
 'Transport-moving']

In [90]:
def hour_group(h):
    if h < 35:
        return 'Low'
    elif 35 <= h <= 45:
        return 'Normal'
    else:
        return 'High'

df_train_test['hours_group'] = df_train_test['hours_per_week'].apply(hour_group)


In [91]:
df_train_test['occ_hours'] = (
    df_train_test['occupation'] + " | " + df_train_test['hours_group']
)


In [92]:
df_train_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,hours_group,occ_hours
0,39,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K,train,Normal,Adm-clerical | Normal
1,50,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K,train,Low,Exec-managerial | Low
2,38,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K,train,Normal,Handlers-cleaners | Normal
3,53,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K,train,Normal,Handlers-cleaners | Normal
4,28,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K,train,Normal,Prof-specialty | Normal


In [93]:
pd.crosstab(df_train_test['occupation'], df_train_test['hours_group'])


hours_group,High,Low,Normal
occupation,,,
Adm-clerical,494,1023,4094
Armed-Forces,5,1,9
Craft-repair,1286,383,4443
Exec-managerial,2268,397,3421
Farming-fishing,635,212,643
Handlers-cleaners,238,438,1396
Machine-op-inspct,408,213,2401
Other-service,474,1833,2616
Priv-house-serv,29,119,94


In [94]:
pd.crosstab(
    [df_train_test['occupation'], df_train_test['hours_group']],
    df_train_test['income']
)


income                         <=50K  >50K
occupation        hours_group             
Adm-clerical      High           364   130
                  Low            959    64
                  Normal        3520   574
Armed-Forces      High             3     2
                  Low              1     0
                  Normal           6     3
Craft-repair      High           865   421
                  Low            365    18
                  Normal        3499   944
Exec-managerial   High           861  1407
                  Low            317    80
                  Normal        2000  1421
Farming-fishing   High           522   113
                  Low            205     7
                  Normal         590    53
Handlers-cleaners High           210    28
                  Low            434     4
                  Normal        1290   106
Machine-op-inspct High           328    80
                  Low            210     3
                  Normal        2112   289
Other-service     High           421    53
                  Low           1804    29
                  Normal        2494   122
Priv-house-serv   High            29     0
                  Low            118     1
                  Normal          92     2
Prof-specialty    High           746  1039
                  Low            631   170
                  Normal        2011  1575
Protective-serv   High           131   106
                  Low             88     5
                  Normal         456   197
Sales             High           913   712
                  Low           1133    65
                  Normal        1983   698
Tech-support      High           129    79
                  Low            168    29
                  Normal         729   312
Transport-moving  High           547   223
                  Low            213    16
                  Normal        1114   242

In [95]:
import pandas as pd
import scipy.stats as stats

# Contingency table
table = pd.crosstab(
    [df_train_test['occupation'], df_train_test['hours_group']],
    df_train_test['income']
)

# Chi-square test
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)


Chi-square: 7382.081964766668
Degrees of freedom: 41
p-value: 0.0


In [96]:
chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("Degrees of freedom:", dof)
print("p-value:", p)


Chi-square: 7382.081964766668
Degrees of freedom: 41
p-value: 0.0


<span style="color: blue;">Interpretation:
p‑value = 0.0
This does not mean the p‑value is literally zero, The p‑value is so extremely small that Python rounds it to 0.0.
i.e. This indicates overwhelming evidence against the null hypothesis. </span>

<span style="color: blue;">We Reject H₀.
There is a highly significant association between: Occupation ,Hours‑per‑week , Income class (>50K vs ≤50K) </span>

<span style="color: blue;">Occupation, Hours‑per‑week, Income class (>50K vs ≤50K)
i.e. Certain occupations combined with specific working‑hour patterns strongly influence the probability of earning >50K. </span>

<span style="color: blue;">Type I Error (False Positive)
We conclude that occupation × hours affects income,
even though in reality it does NOT.
i.e. certain occupations + long hours increase income, but they actually don’t. </span>

<span style="color: blue;">Type II Error (False Negative)
We conclude that occupation × hours does NOT affect income,
even though in reality certain occupations with long hours DO increase income probability.
i.e.We miss a real effect. </span>

In [97]:
df_train_test.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,source,hours_group,occ_hours
0,39,State-gov,77516.0,Bachelors,13.0,Never-married,Adm-clerical,Not-in-family,White,Male,2174.0,0.0,40.0,United-States,<=50K,train,Normal,Adm-clerical | Normal
1,50,Self-emp-not-inc,83311.0,Bachelors,13.0,Married-civ-spouse,Exec-managerial,Husband,White,Male,0.0,0.0,13.0,United-States,<=50K,train,Low,Exec-managerial | Low
2,38,Private,215646.0,HS-grad,9.0,Divorced,Handlers-cleaners,Not-in-family,White,Male,0.0,0.0,40.0,United-States,<=50K,train,Normal,Handlers-cleaners | Normal
3,53,Private,234721.0,11th,7.0,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0.0,0.0,40.0,United-States,<=50K,train,Normal,Handlers-cleaners | Normal
4,28,Private,338409.0,Bachelors,13.0,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0.0,0.0,40.0,Cuba,<=50K,train,Normal,Prof-specialty | Normal


# Hypothesis question : Age vs Income

In [98]:
# hypothesis question
# Does age affect the probability of Income
# H0 - Age does not affect the probability of earning > 50K
# H1 - Age significantly affects the probability of earngin > 50K

# 1 = >50K, 0 = <=50K
df_train_test['age'] = pd.to_numeric(df_train_test['age'], errors='coerce')
df_train_test['income_binary'] = (df_train_test['income'] == '>50K').astype(int)

model = smf.logit (formula = "income_binary ~ age",data=df_train_test)
result = model.fit()
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.524280
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:          income_binary   No. Observations:                48842
Model:                          Logit   Df Residuals:                    48840
Method:                           MLE   Df Model:                            1
Date:                Mon, 09 Feb 2026   Pseudo R-squ.:                 0.04720
Time:                        11:44:15   Log-Likelihood:                -25607.
converged:                       True   LL-Null:                       -26875.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.7228      0.035    -78.308      0.000      -2.791      -2.655
age            0.0387      0.

In [99]:
np.exp(0.0387)

1.039458599290056

<span style="color: blue;">
- Convert logistic regression coefficient from log-odds to odds ratio.</span>

<span style="color: blue;"> - In logistic regression, coef represents change in log-odds of the outcome per unit increase in predictor.</span>

<span style="color: blue;">  - np.exp(coef) gives the odds ratio, which is easier to interpret: </span>

<span style="color: blue;">   - OR > 1 means higher odds of outcome as predictor increases.</span>

<span style="color: blue;">   - OR < 1 means lower odds.</span>

<span style="color: blue;"> - Example: coef = 0.0387 → odds ratio ≈ 1.0395 --> each 1-year increase in age raises odds of earning >50K by ~3.95%.
</span>

 <span style="color: blue;"><b> Results :</b></span>
 <span style="color: blue;"> Age coefficient = 0.0387 - Each year of age increases the probability of earning > 50K by about 3.95%.</span>
 <span style="color: blue;">Pseudo R² = 0.0472 - Age alone explains ~4.7% of the variation in income.</span>

 <span style="color: blue;">p-value for age = 0.000  < 0.05 --> reject H0 null hypothesis. age is statistically significant.</span>

 <span style="color: blue;">Conclusion : age is significant preditor of income, older individuals having higher probability of earning > 50K
</span>

# Hypothesis : Race vs Income
#### H0: Race has no effect on income (person’s race does not change the likelihood of earning >50K or ≤50K).
#### H1 : Race has an effect on income (person’s race changes the likelihood of earning >50K or ≤50K).

In [100]:

# Create a contingency table
contingency = pd.crosstab(df_train_test['race'], df_train_test['income'])

# Perform Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)

# Print results
print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Expected frequencies:\n", expected)

# Interpretation
if p < 0.05:
    print("Reject H0: Race and income are not independent (significant relationship).")
else:
    print("Fail to reject H0: No significant relationship between race and income.")


Chi-square statistic: 487.026286837627
p-value: 4.284377710223499e-104
Degrees of freedom: 4
Expected frequencies:
 [[  357.53757012   112.46242988]
 [ 1155.53099791   363.46900209]
 [ 3563.96492773  1121.03507227]
 [  308.85160313    97.14839687]
 [31769.11490111  9992.88509889]]
Reject H0: Race and income are not independent (significant relationship).


#### - Degrees of freedom tell us how many independent pieces of information are used to calculate the Chi-square statistic.
#### - dof=(number of rows−1)×(number of columns−1)
#### -      Rows = number of categories in one variable (e.g., races)
#### -      Columns = number of categories in the other variable (e.g., income groups)

<span style="color: blue;"><b> Race vs Income Analysis Results</b></span>

<span style="color: blue;"> -The Chi-square test shows a very strong association between race and income (Chi-square = 487.03, df = 4, p < 0.001).</span>

<span style="color: blue;"> -The expected frequencies (the counts we would expect if race had no effect on income) differ from the actual observed counts, which is why the test is highly significant.</span>

<span style="color: blue;"> -Conclusion: Race has a significant effect on income — the likelihood of earning >50K varies across different racial groups. </span>

# Hypothesis : Marital status vs Income
#### - H0 : Income level is independent of marital status
#### - H1 : Income level is dependednt of marital status

In [101]:
# Create contingency table
income_cont = pd.crosstab(df_train_test['marital_status'], df_train_test['income'])

#chi-square test
chi2, p, dof, expected = chi2_contingency(income_cont)

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)

Chi-square statistic: 9816.015037266438
p-value: 0.0
Degrees of freedom: 6


<span style="color: blue;"><b> Results: - Since the p-value is less than 0.05, we reject the null hypothesis. </b></span>

<span style="color: blue;">- There is a statistically significant association between marital status and income level.</span>

<span style="color: blue;"> - Income (>50K vs ≤50K) differs significantly across marital-status categories.</span>

# Hypothesis : Country and education levels association with Income

In [102]:
# First, define the groups
low_edu = [
    'Preschool','1st-4th','5th-6th','7th-8th',
    '9th','10th','11th','12th','HS-grad'
]

high_edu = [
    'Bachelors','Prof-school','Assoc-acdm',
    'Assoc-voc','Masters','Doctorate'
]

# Create a new column in the dataframe
df_train_test['education_group'] = df_train_test['education'].apply(
    lambda x: 'low_edu' if x in low_edu else ('high_edu' if x in high_edu else 'Other')
)
#print(df_train_test['education_group'].value_counts())
model = smf.logit(
    "income_binary ~ C(education_group) + C(native_country)",
    data=df_train_test
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.500310
         Iterations 17
                           Logit Regression Results                           
Dep. Variable:          income_binary   No. Observations:                47985
Model:                          Logit   Df Residuals:                    47942
Method:                           MLE   Df Model:                           42
Date:                Mon, 09 Feb 2026   Pseudo R-squ.:                 0.09016
Time:                        11:44:25   Log-Likelihood:                -24007.
converged:                       True   LL-Null:                       -26386.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                      coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------------------
Intercept                        

<span style="color: blue;"><b> Results:  Higher education significantly increases the likelihood of earning more than 50K, regardless of country.</b></span>

<span style="color: blue;">-Country also affects income probabilities, with some countries showing higher or lower odds.</span>

<span style="color: blue;">- Overall, education is the strongest predictor of income in this dataset.</span>